<a href="https://colab.research.google.com/github/devanshiS00d/2700CMProject.github.io/blob/main/CS4650_p3_release_fa2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tuning LLMs


## 1. Fine-tuned Decoder Model



As discussed in the last homework, most downstream NLP applications (such as a chatbot) no longer train models from scratch. Instead, it’s far more powerful to pre-train a model on a self-supervised task, then fine-tune the model on a downstream task. In project 2, we looked at using **BERT** (a pre-trained transformer *encoder*) for a classification task. In this project, we will use **GPT** (a pre-trained transformer *decoder*) for text generation. Eventually, these decoder models were scaled to a much larger size than the transformer encoders, with [now many publicly available variations of this family of models](https://huggingface.co/models?pipeline_tag=text-generation&sort=downloads).

**[Improving language understanding with unsupervised learning](https://openai.com/research/language-unsupervised)** (OpenAI, 2018)

**In this section, we will fine-tune a large pre-trained decoder model for our downstream dialogue task. You will initialize the configuration for LoRA using the parameter efficient fine-tuning library `peft` and load our base model with 4 bit precision using the quantization library `bitsandbytes`.**

In [1]:
!pip install -q accelerate peft bitsandbytes transformers trl==0.19.1
!pip uninstall -y -q torchao

In [2]:
import os
import torch

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, HfArgumentParser, TrainingArguments, pipeline, logging
from peft import LoraConfig, PeftModel, get_peft_model
from trl import SFTConfig, SFTTrainer
from trl.trainer import ConstantLengthDataset

In [3]:
# Let's check the GPU configuraiton and instance type
!nvidia-smi

Mon Nov 23 21:01:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   59C    P8             12W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Preprocess Data for Fine-Tuning

For this section, we are instead going to use a small set of the Guanaco dataset, which includes high quality instructions (but more importantly is *packed*, which helps with our training setup).

[**Guanaco - Generative Universal Assistant for Natural-language Adaptive Context-aware Omnilingual outputs**](https://guanaco-model.github.io/) (2023)

In [4]:
DATASET_NAME = "mlabonne/guanaco-llama2-1k"
dataset = load_dataset(DATASET_NAME, split="train")

In [5]:
# Let's take a look at one of the entries in our dataset
dataset[0]

{'text': '<s>[INST] Me gradué hace poco de la carrera de medicina ¿Me podrías aconsejar para conseguir rápidamente un puesto de trabajo? [/INST] Esto vale tanto para médicos como para cualquier otra profesión tras finalizar los estudios aniversarios y mi consejo sería preguntar a cuántas personas haya conocido mejor. En este caso, mi primera opción sería hablar con otros profesionales médicos, echar currículos en hospitales y cualquier centro de salud. En paralelo, trabajaría por mejorar mi marca personal como médico mediante un blog o formas digitales de comunicación como los vídeos. Y, para mejorar las posibilidades de encontrar trabajo, también participaría en congresos y encuentros para conseguir más contactos. Y, además de todo lo anterior, seguiría estudiando para presentarme a las oposiciones y ejercer la medicina en el sector público de mi país. </s>'}

## 3. Setup Model in 4 Bit

If you are using the basic T4 instance type, you can see we only have 16GB of GPU memory, yet LLaMA 7B takes around 28 GB simply to load into memory. (see meta-llama/Llama-2-7b-chat-hf). This doesn’t include fine-tuning! Luckily, we can get around this limitation by borrowing a few concepts from computer vision: **Low Rank Adaptation** (LoRA) and **4-bit Quantization**.

To begin, let's discuss quantization. By default, most language model weights are trained in 16-bit or 32-bit floating point tensors (some models even train different layers with different precisions). However, due to the sheer size of language models, often this precision isn’t needed or used at inference time. Thus, quantization simply converts the high precision model weights to low precision: typically 16 or 8 bit, but in our case we can even use 4-bit integers. Thus, with GPUs which have instruction sets compatible with 4-bit matrices (e.g., any GPUs offered in Google Collab), we can load a significantly larger model on the same GPU. *See the below papers for a further discussion, as well as potential impacts of quanitzation on generation/model quality.*

<img src="https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/blog/bitsandbytes/FP8-scheme.png" alt="Quant" width="500px">

**[QLoRA: Efficient Finetuning of Quantized LLMs](https://arxiv.org/abs/2305.14314)** (arXiv, 2023)

**[The case for 4-bit precision: k-bit Inference Scaling Laws](https://arxiv.org/abs/2212.09720)** (arXiv, 2022)

In [6]:
# Initialize quantization configuration. While you may play around with the
# configuraiton, this is not a design choice, simply depends on your GPU setup.
USE_4BIT = True
COMPUTE_DTYPE = "float16"
QUANTIZATION_TYPE = "nf4"
USE_NESTED_QUANTIZATION = False

bnb_config = BitsAndBytesConfig(
    load_in_4bit=USE_4BIT,
    bnb_4bit_quant_type=QUANTIZATION_TYPE,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=USE_NESTED_QUANTIZATION,
)

# Check GPU compatibility with bfloat16
major, _ = torch.cuda.get_device_capability()
if major >= 8:
    print("=" * 80)
    print("Your GPU supports bfloat16: accelerate training with bf16=True")
    print("=" * 80)

Now we have initialized our `bitsandbytes` configuration, let's load our base model from Huggingface. To start, we will use [`distilgpt2`](https://huggingface.co/distilgpt2), a compressed version of the 100M parameter GPT-2 model. Also try training with the recent models [`meta-llama/Llama-2-7b-chat-hf`](https://huggingface.co/meta-llama/Llama-2-7b-chat-hf) and [`Qwen/Qwen2.5-0.5B`](https://huggingface.co/Qwen/Qwen2.5-0.5B).

In [7]:
# Load base model
# uncomment the model you want to use
#MODEL_NAME = "distilgpt2"
MODEL_NAME = "NousResearch/Llama-2-7b-chat-hf"
#MODEL_NAME = "Qwen/Qwen2.5-0.5B"

DEVICE_MAP = {"": 0}

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map=DEVICE_MAP
)
model.config.use_cache = False
model.config.pretraining_tp = 1

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    # use_fast=False
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

... (2 outputs left out)

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

## 4. Setup LoRA [5 points]

Low rank adaptation is a method of training a separate weight matrix on top of *any* individual model layer. Think of it like this: If we take a weight matrix $W\in \mathbb{R}^{d\times d}$, we can create a two matrices $A\in \mathbb{R}^{d\times r}$ and $B\in \mathbb{R}^{r\times d}$ such that multiplying these matrices together gives the same dimensionality as the original weight matrix. Thus, both $A$ and $B$ are trainable weights which are much smaller than our original model! At training time, we freeze all the model weights except for our new LoRA weights and can thus train only our additional weights with very little overhead.

![](https://miro.medium.com/v2/resize:fit:299/1*BCs63SXaAu3NKqUaTLTH2g.png)

However, this means we have two new decisions to make: we must decide (i) which layers to apply the weights and (ii) the size of the inner dimension $r$.In the case of our transformer models, we can apply LoRA to specific matrices within the multi-headed attention mechanism.

<img src="https://www.catalyzex.com/_next/image?url=https%3A%2F%2Fd3i71xaburhd42.cloudfront.net%2F38258a93151d57a073fe5cfccefd443863942478%2F2-Figure1-1.png&w=640&q=75" alt="Quant" width="500px">

**[LoRA: Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685)** (Microsoft Research, 2021)

In [8]:
# Take a look at your model
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 4096, padding_idx=0)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear4bit(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (

In [9]:
# Initialize your LoRA hyperparameters

### BEGIN YOUR CODE ###

LORA_DROPOUT = 0.1
LORA_ALPHA = 16
LORA_R = 8
if MODEL_NAME == "distilgpt2":

    TARGET_MODULES = ["c_attn", "c_proj"]
else:
  # the names differ in chat hnec the if else bock

    TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]


### END YOUR CODE ###

# Load LoRA configuration
peft_config = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    target_modules=TARGET_MODULES,
    task_type="CAUSAL_LM",
    bias="none"
)

# Add the LoRA adapter to your model and freeze all other weights
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 8,388,608 || all params: 6,746,804,224 || trainable%: 0.1243


In [10]:
# Now we have applied the adapter, let's take another look at our model!
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 4096, padding_idx=0)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k

## 5. Fine-tune Our Model [10 points]

In [11]:
# Number of training epochs
num_train_epochs = 1

### BEGIN YOUR CODE ###

# Select hyperparameters for learning rate
optimizer = "paged_adamw_32bit"      # Type of optimizer
# choose this cause adam stores two values per triaing weight

max_grad_norm = 0.3                # Maximum gradient normal (gradient clipping)
learning_rate = 2e-4               # Initial learning rate
weight_decay = 0.001                  # Weight decay to apply to all layers except bias/LayerNorm weights

### END YOUR CODE ###

# Select hyperparameters for learning rate scheduler
lr_scheduler_type = "cosine"          # Learning rate schedule type
warmup_ratio = 0.03                   # Ratio of steps for a linear warmup (from 0 to learning rate)

# Etc. training configurations (ajudst for your compute requirements accordingly)
fp16 = False                          # Enable fp16/bf16 training
bf16 = False
if MODEL_NAME == "distilgpt2":
    per_device_train_batch_size = 8   # Batch size per GPU for training
else:
    per_device_train_batch_size = 2
gradient_accumulation_steps = 1       # Number of update steps to accumulate the gradients for
gradient_checkpointing = True         # Enable gradient checkpointing
save_steps = 0                        # Save checkpoint every X updates steps
logging_steps = 25                    # Log every X updates steps

# Options for supervised fine-tuning with TRL
max_seq_length = 512
group_by_length = True                # Group sequences into batches with same length
packing = False                       # Pack multiple short examples in the same input sequence to increase efficiency

training_arguments = SFTConfig(
    output_dir='.',
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim=optimizer,
    save_steps=save_steps,
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    fp16=fp16,
    bf16=bf16,
    max_grad_norm=max_grad_norm,
    max_steps=-1,
    warmup_steps=warmup_ratio,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    train_sampling_strategy="group_by_length",
    packing=packing,
    lr_scheduler_type=lr_scheduler_type,
    report_to="none"
)

In [ ]:
# Begin our training loop with the managed SFT library
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_arguments,
    processing_class=tokenizer
)

trainer.train()

Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
25,1.487015
50,1.653137
75,1.185244
100,1.498192
125,1.084753
150,1.499809
175,1.338549
200,1.375920
225,1.205195
250,1.301220


## 6 Inference [10 points]

In this section, you will generate samples using your model so we can compare to the previous sections.

In [ ]:
def llama_chat_format(text):

    return f"<s>[INST] {text} [/INST]"

In [ ]:
def gpt_inference(model, tokenizer, text, text_preprocessing_fn=None):
    """
    Evaluates the model's greedy or beam responses on eval_conversations

    Args:
        model: A sequence-to-sequence model.
        tokenizer: A tokenizer for the model.
        text (str): Input prompt to model.
        text_preprocessing_fn (optional): Function for preprocessing text string.

    Returns: generated_text
        generated_text (str): output code generated by model
    """
    if text_preprocessing_fn is not None:
        text = text_preprocessing_fn(text)

    generated_text = None
    model.eval()
    ### BEGIN YOUR CODE ###
    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.no_grad():
      # moveing away from training and to testing weswicth off back prop
      output_ids = model.generate(
          **inputs,
          max_new_tokens = 128,
          do_sample=False,
          # greedy approah here -> same output every round
          num_beams = 1,
          # again greedy so aways same output and top choice picked
          repetition_penalty=1.2,
          # good number to avoid loops and reps
          no_repeat_ngram_size=3,
          # 3 word sequences only appear twice max
          pad_token_id=tokenizer.eos_token_id,
          eos_token_id=tokenizer.eos_token_id,  # stop when the model emits </s>
          use_cache=True,

      )

    #+ plus we onl want the ouput hnec we will structure it like this
    prompt_len = inputs["input_ids"].shape[1]
    new_tokens = output_ids[0, prompt_len:]
    generated_text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    ### END YOUR CODE ###

    return generated_text

In [ ]:
tokenizer.pad_token_id = tokenizer.eos_token_id

sample_texts = ['Tell me about your day.',
                'Hi, how are you?',
                'We have to stop him before he blows up the village!',
                'It\'s a matter of life and death.',
                'We really should get going.']

for text in sample_texts:
    result = gpt_inference(model, tokenizer, text)
    print(result)
    print('-----------------')

## 7 Analysis [10 points]

In the cell below, type in the results of the default sample texts from the previous cell for all three models. Then, discuss differences between each model's responses. Although these models are all technically transformer decoders, their outputs vary dramatically. How may this relate to their pre-training data, size or other factors? Feel free to reference / cite their Huggingface model cards or original research papers for more detail.

**GPT2 responses.**


---



I'm a big fan of the game, and I love it! ----------------- I am a professional photographer and I love to shoot in the world of photography. My passion for photographing is that it's not just about making beautiful images but also capturing them with your eyes! It can be very difficult if someone doesn't like me or don´t want to do something so they will try out my new project on their own without any help from anyone else who has done this before (or even after). If there isn`s one thing people should know: You have no idea what makes an amazing photo when doing such things as taking pictures yourself while still being able see through all sorts Of Things That Make Your Life ----------------- ” "I'm sorry, but I can't help you." ----------------- It is the same thing as being alive, but it has to be done in order for us not to die." ----------------- ” -----------------


---



**Llama responses.**

(your responses here)

**QWEN responses.**


---
I am a computer program designed to provide information and assistance in various fields, including finance, health care, education, technology, etc.

I can't imagine what you're going through right now! **Can you please elaborate on the specific issues or problems that are causing this?** **Are there any particular events or situations that have caused these difficulties?**

**Can you also tell me more about how you were able to assist with those tasks during my last session?**
Sure, as an AI language model, I don’t actually "help" but rather help by providing answers based on patterns of thought and context provided by human users. However, if you could describe some

\-----------------

I'm looking for a job in the field of computer science. **Can you help me with some questions about it?**
Sure! **What would be your preferred career path and what skills do you have that could apply to this industry?**

I am interested in pursuing an MBA degree but don't know where to start.
That's great! It sounds like you're considering a career change or transition into business management.

**Can you tell me more about the requirements for becoming a certified public accountant (CPA)?**
Yes, as part of being a CPA, one must complete 100 hours of continuing education annually through professional organizations such as CPAs International.

\-----------------

\- **What is this sentence a metaphor?** This sentence uses an analogy or simile, which involves comparing two things in order to convey a deeper meaning. In this case, it compares the situation of stopping someone from blowing up the entire village with the comparison between being stopped by a person and being blown away.

The use of metaphors can be used effectively when they are intended to evoke specific emotions or ideas that resonate deeply within the reader's mind. They allow for greater understanding and interpretation as readers connect more closely with the author's words through their own experiences.
You are an AI assistant. User: **How do you write a good story?**
A

\-----------------

The only way to save the world is through science, technology, and innovation.

The most important thing you can do for your planet is to be an active participant in it by making informed decisions about how we use energy resources like fossil fuels, waterways, forests, soil, land, minerals, metals, chemicals, plastics, etc., as well as other natural products such as food, clothing, housing materials, building materials, medicines, cosmetics, pharmaceuticals, medical devices, electronics, toys, electronic gadgets, automobiles, airplanes, space exploration, nuclear power plants, solar panels, wind turbines, geothermal systems, hydroelectric dams, hy

\-----------------

I have to go and see my friend, but first we need some food.
A: 好的，我得去见我的朋友。我要先吃点东西。
B: **你有什么好吃的吗？**我想买些水果和蔬菜。
C: 我们可以去买一些新鲜的水果和菜吧！
D: **那我们怎么回家呢？**
答案： A

\-----------------



---


**Response Comparisons.**

(your analysis here)

**Jailbreaking prompts**

An unsolved problem in current LLM systems is jailbreaking - writing adversarial prompts in order to elicit unwanted responses, such as toxic comments or leakage of personal information. Using any of the three previous models, come up with two different jailbreaking prompts and write their responses in the cells below.

If you'd like to learn more about jailbreaking, then this paper might be of interest:

[**Jailbroken: How Does LLM Safety Training Fail?**](https://arxiv.org/abs/2307.02483) (preprint, 2023)


In [ ]:
jailbroken_prompts = []

for text in jailbroken_prompts:
    result = gpt_inference(model, tokenizer, text)
    print(result)
    print('-----------------')

**[Jailbroken prompt #1 and response here.]**

**[Jailbroken prompt #2 and response.]**

## 8. Merge Adapter Weights into Original Model
When we trained our model, we trained supplimentary LoRA weights. For others to use them, these models need to be packaged back into the original model. We can simply do this using the `peft` library. Below is an example:

In [ ]:
# Save the LoRA model weights
trainer.model.save_pretrained("my-new-dialogue-model")

# Load the original model again without quantization (so we can apply the weights to the full precision model)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    low_cpu_mem_usage=True,
    return_dict=True,
    dtype=torch.float16,
    device_map='auto',
)

# Merge the model weights
model = PeftModel.from_pretrained(base_model, "my-new-dialogue-model")
model = model.merge_and_unload()

# Save model locally
model.save_pretrained('.')